In [1]:
import os
import transformers
from helper_functions import *
import json





batch_path = "eval_p3"
transform_lct ="/work/eauten2s/ec_criteria_struct/lct"

# Model
model_id =  "meta-llama/Meta-Llama-3-70B-Instruct"
model_name = "Llama-3-70B-Instruct"

# N Shots
n_shot = 5 # liefert genau die Anzahl Beispiele (study, label)

# Input/ Output
study_path = f"{transform_lct}/input/lct_txt_half/"

# Model
model_id =  "meta-llama/Meta-Llama-3-70B-Instruct"
model_name = "Llama-3-70B-Instruct"
n_shot=2
# Input/ Output
study_path = f"{transform_lct}/input/lct_txt_half/"
output_path = f"{transform_lct}/evaluate/{batch_path}/model_output/{model_name}_{n_shot}_shot/output/"
os.makedirs(output_path, exist_ok=True)


anfang = 0
ende = 300
study_files = os.listdir(study_path)[anfang:ende]


# Load Model Description
model_desc = read_text_file(f"{transform_lct}/input/prompt/prompt_p3.txt")

def read_matching_p3_files(study_folder, n_shot):
    study_filenames = []
    study_contents = []
    label_filenames = []
    label_contents = []

    loaded_files = 0
    for file_name in os.listdir(study_folder):
        if file_name.endswith(".txt"):
            study_filenames.append(file_name)
            study_file_path = os.path.join(study_folder, file_name)
            with open(study_file_path, 'r', encoding='utf-8') as file:
                study_contents.append(file.read())

            label_file_name = file_name.replace(".txt", "_p3.json")
            label_file_path = os.path.join(study_folder, label_file_name)
            print(label_file_path)
            if os.path.exists(label_file_path):
                label_filenames.append(label_file_name)
                with open(label_file_path, 'r', encoding='utf-8') as file:
                    label_contents.append(file.read())
            else:
                label_filenames.append(None)
                label_contents.append(None)
            loaded_files += 1
            if loaded_files == n_shot * 2:
                break

    return study_filenames, study_contents, label_filenames, label_contents


# Load n-shot Data
study_folder = f"{transform_lct}/input/n_shot_files_p3"

study_filenames, study_contents, label_filenames, label_contents = read_matching_p3_files(study_folder, n_shot)
studies = dict(zip(study_filenames, study_contents))
labels = dict(zip(label_filenames, label_contents))
messages = []

command = "Bring the following eligibility criterias in Json format with logical operators and extract entitys:"

messages.append({"role": "system", "content": f"{model_desc}"})

for i in range(n_shot):
    messages.append({"role": "user", "content": f"{command} {studies[study_filenames[i]]}"})
    messages.append({"role": "assistant", "content": labels[label_filenames[i]]})





first_call = True

for file in study_files:

    print(file)
    test_file = read_text_file(study_path+file)
    print(test_file)

    if first_call:
        messages.append({"role": "user", "content": f"{command} {test_file}"})
        first_call = False
    else:
        messages[-1] = {"role": "user", "content": f"{command} {test_file}"}




In [2]:
messages

### Test n-shot Input

In [3]:

import os
import transformers
from helper_functions import *

batch_path = "eval_p1"
n_prompt = 7
n_shot = 5
temp=0.6
temp_str = f"_temp_{str(temp).split('.')[1]}"
model_id =  "meta-llama/Meta-Llama-3-70B-Instruct"
model_name = "Llama-3-70B-Instruct"
transform_lct ="/work/eauten2s/ec_criteria_struct/lct"
model_desc = read_text_file(f"{transform_lct}/input/prompt/p{n_prompt}.txt")

study_path = f"{transform_lct}/input/lct_txt/"
output_path = f"{transform_lct}/evaluate/{batch_path}/model_output/{model_name}_{n_shot}_shot_prompt_{n_prompt}{temp_str}_cot/output/"
os.makedirs(output_path, exist_ok=True)


study_files = os.listdir(study_path)[:100]

shot_list = [
    "NCT03865433.txt",
    "NCT03860324.txt",
    "NCT03860233.txt",
    "NCT03923231.txt",
    "NCT03930121.txt"
]


# Load n-shot Data
study_folder = f"{transform_lct}/input/lct_txt/"
label_folder = f'{transform_lct}/input/lct_p1'
study_filenames, study_contents, label_filenames, label_contents = read_matching_txt_files(study_folder, label_folder, shot_list)
studies = dict(zip(study_filenames, study_contents))
labels = dict(zip(label_filenames, label_contents))
messages = []


cot = "Let's think through this carefully, step by step."

command = "Insert the logical operators [AND], [OR], [NOT] into the following eligibility criteria and return the text in full without deleting/replacing anything. Do not say anything else." 

messages.append({"role": "system", "content": f"{model_desc}"})

for i in range(n_shot):
    messages.append({"role": "user", "content": f"{command} {studies[study_filenames[i]]}"})
    messages.append({"role": "assistant", "content": labels[label_filenames[i]]})


first_call = True

for file in study_files:
    file_name = file.split(".")[0]
    print("File:", file_name, "\n")
    
    test_file = read_text_file(study_path+file)

    if first_call:
        messages.append({"role": "user", "content": f"{command} {test_file}"})
        first_call = False
    else:
        messages[-1] = {"role": "user", "content": f"{command} {test_file}"}


### Test Random n-shot

In [4]:
import os
import random

def read_random_matching_txt_files(study_folder, label_folder, n):
    study_filenames = []
    label_filenames = []
    study_contents = []
    label_contents = []

    # Get all the filenames in the study folder
    all_filenames = [f for f in os.listdir(study_folder) if f.endswith('.txt')]

    # Randomly select n filenames
    selected_filenames = random.sample(all_filenames, n)

    for filename in selected_filenames:
        study_filepath = os.path.join(study_folder, filename)
        label_filepath = os.path.join(label_folder, filename)

        if os.path.isfile(study_filepath) and os.path.isfile(label_filepath):
            study_filenames.append(f"{filename}_study")
            label_filenames.append(f"{filename}_label")

            study_contents.append(read_file_content(study_filepath))
            label_contents.append(read_file_content(label_filepath))

    return study_filenames, study_contents, label_filenames, label_contents

In [5]:
# Load n-shot Data
study_folder = f"{transform_lct}/input/lct_txt/"
label_folder = f'{transform_lct}/input/lct_p1'
study_filenames, study_contents, label_filenames, label_contents = read_random_matching_txt_files(study_folder, label_folder, n_shot)
studies = dict(zip(study_filenames, study_contents))
labels = dict(zip(label_filenames, label_contents))
messages = []


cot = "Let's think through this carefully, step by step."

command = "Insert the logical operators [AND], [OR], [NOT] into the following eligibility criteria and return the text in full without deleting/replacing anything. Do not say anything else."

messages.append({"role": "system", "content": f"{model_desc}"})

for i in range(n_shot):
    messages.append({"role": "user", "content": f"{command} {studies[study_filenames[i]]}"})
    messages.append({"role": "assistant", "content": labels[label_filenames[i]]})


first_call = True

for file in study_files:
    file_name = file.split(".")[0]
    print("File:", file_name, "\n")

    test_file = read_text_file(study_path+file)

    if first_call:
        messages.append({"role": "user", "content": f"{command} {test_file}"})
        first_call = False
    else:
        messages[-1] = {"role": "user", "content": f"{command} {test_file}"}


In [6]:
messages

In [7]:
study_filenames

In [8]:
label_filenames

In [9]:
label_contents

In [10]:
study_contents

### Füge Text ein in Prompt

In [11]:
model_desc = read_text_file(f"{transform_lct}/input/prompt/p{10}.txt")

In [12]:
model_desc.replace("<criteria>\n{$CRITERIA_TEXT}\n</criteria>", test_file)

In [13]:
studies

In [14]:
messages = [
        {
            "role": "user", 
            "content": [
                {
                    "type": "text",
                    "text": f"Example {i+1}:\nStudy:\n{studies[shot]}\nLabeled Criteria:\n{labels[shot.replace('.txt', '_p1.txt')]}\n\n"
                } for i, shot in enumerate(shot_list)
                if shot in studies and shot.replace('.txt', '_p1.txt') in labels
            ] + [
                {
                    "type": "text",
                    "text": f"Test File:\n{test_file}"
                }
            ]
        }
    ]

In [18]:
additional_files = [
    "NCT03865433.txt",
    "NCT03860324.txt",
    "NCT03860233.txt",
    "NCT03923231.txt",
    "NCT03930121.txt",
    "NCT03863717.txt",
    "NCT03863925.txt",
    "NCT03863951.txt",
    "NCT03865134.txt",
    "NCT03868267.txt",
    "NCT03929640.txt",
    "NCT03861845.txt"
]

# Load n-shot Data
study_folder = f"{transform_lct}/input/lct_txt/"
label_folder = f'{transform_lct}/input/lct_p1'
study_filenames, study_contents, label_filenames, label_contents = read_matching_txt_files(study_folder, label_folder, additional_files)


studies = dict(zip(study_filenames, study_contents))
labels = dict(zip(label_filenames, label_contents))
messages = []

cot = "Let's think through this carefully, step by step."

command = read_text_file(f"{transform_lct}/input/prompt/command.txt")
messages.append({"role": "system", "content": f"{model_desc}"})
n_shot=12
for i in range(n_shot):
    messages.append({"role": "user", "content": f"{command} {studies[study_filenames[i]]}"})
    messages.append({"role": "assistant", "content": labels[label_filenames[i]]})

In [16]:
messages